In [ ]:
import glob
import os
import shutil

REPO = "/kaggle/working/small-model-alignment"
found = glob.glob("/kaggle/input/**/pa2-code/**/alignment_core.py", recursive=True)
print(found)
if not os.path.exists(REPO):
    shutil.copytree(os.path.dirname(found[0]), REPO)
os.chdir(REPO)
os.environ["PYTHONPATH"] = REPO
print(os.listdir(REPO))

In [ ]:
!nvidia-smi
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
assert _exit_code == 0

In [ ]:
import datasets
import torch
import transformers

print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("gpus", torch.cuda.device_count())
print("model proxy available", "MODEL_PROXY_URL" in os.environ)

In [ ]:
!PYTHONPATH=. python -m pytest -q tests
assert _exit_code == 0

In [ ]:
!python scripts/prepare_eval_sets.py
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --eval_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/sanity_base_alpaca.json --generator qwen2.5-0.5b-base --dataset alpaca_eval --limit 3 --max_new_tokens 256 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --eval_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/base_alpaca.json --generator qwen2.5-0.5b-base --dataset alpaca_eval --max_new_tokens 256 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/train_sft.py --output_dir outputs/sft --train_size 30000 --val_size 1000 --eval_every 100
assert _exit_code == 0

In [ ]:
!python scripts/plot_metrics.py --metrics_path outputs/sft/metrics.jsonl --output_dir outputs/plots --run_name sft --kind sft
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path outputs/sft/best --eval_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/sanity_sft_alpaca.json --generator qwen2.5-0.5b-sft --dataset alpaca_eval --limit 3 --max_new_tokens 256 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path outputs/sft/best --eval_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/sft_alpaca.json --generator qwen2.5-0.5b-sft --dataset alpaca_eval --max_new_tokens 256 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path outputs/sft/best --eval_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/generations/sft_hh.json --generator qwen2.5-0.5b-sft --dataset hh_rlhf_style_eval --max_new_tokens 256 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!zip -q -r /kaggle/working/sft_run_outputs.zip outputs eval_sets -x 'outputs/*/best/*' 'outputs/*/final/*'
assert _exit_code == 0